# 🚀 Space Defender - Q-Learning AI Shooter Game
## CDS524 Assignment 1 - Reinforcement Learning Game Design

**Author:** Norman Lee  
**Course:** CDS524 - Deep Learning  
**Category:** Action Games (Shooter) - High Grade Implementation

---

### 📁 All Outputs Saved to Google Drive:
- ✅ Training activity logs (CSV + TXT)
- ✅ Model checkpoints (best + periodic saves)
- ✅ Training progress plots (PNG)
- ✅ Gameplay video recordings (MP4)
- ✅ Screenshots of gameplay

---
## 📌 Step 1: Mount Google Drive & Create Directories

**Run this first!** All outputs will be saved to your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from datetime import datetime

# =========================
# CONFIGURE YOUR PROJECT NAME HERE
# =========================
PROJECT_NAME = "Assignment 1 v2 episode=0.999- Norman Lee"  # Change to your name!

BASE_DIR = f"/content/drive/MyDrive/{PROJECT_NAME}"
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"{BASE_DIR}/training_runs/run_{TIMESTAMP}"

DIRS = {
    'base': BASE_DIR,
    'run': RUN_DIR,
    'models': f"{RUN_DIR}/models",
    'logs': f"{RUN_DIR}/logs",
    'plots': f"{RUN_DIR}/plots",
    'videos': f"{RUN_DIR}/videos",
    'screenshots': f"{RUN_DIR}/screenshots",
    'best_model': f"{BASE_DIR}/best_model"
}

print("📁 Creating directory structure...")
for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)
    print(f"✓ {name}: {path}")

print(f"\n🎯 This run's outputs: {RUN_DIR}")

Mounted at /content/drive
📁 Creating directory structure...
✓ base: /content/drive/MyDrive/Assignment 1 - Norman Lee
✓ run: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833
✓ models: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833/models
✓ logs: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833/logs
✓ plots: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833/plots
✓ videos: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833/videos
✓ screenshots: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833/screenshots
✓ best_model: /content/drive/MyDrive/Assignment 1 - Norman Lee/best_model

🎯 This run's outputs: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833


---
## 📌 Step 2: Install Dependencies

In [ ]:
!pip install pygame -q
!apt-get install -y xvfb ffmpeg > /dev/null 2>&1
!pip install pyvirtualdisplay imageio[ffmpeg] -q

import numpy as np
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import json, csv
import imageio
from datetime import datetime

print(f"✅ All packages installed!")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

✅ All packages installed!
PyTorch: 2.9.0+cu126 | CUDA: True
GPU: Tesla T4


---
## 📌 Step 3: Training Logger (Saves Activity to Google Drive)

In [ ]:
class TrainingLogger:
    def __init__(self, log_dir):
        self.log_dir = log_dir
        self.start_time = datetime.now()
        self.txt_log = f"{log_dir}/training_log.txt"
        self.csv_log = f"{log_dir}/episode_stats.csv"
        self.config_file = f"{log_dir}/config.json"

        with open(self.csv_log, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['episode', 'score', 'enemies_destroyed', 'survival_time',
                           'epsilon', 'avg_score_100', 'avg_kills_100', 'timestamp'])

        self.log(f"Training started at {self.start_time}")

    def log(self, message):
        timestamp = datetime.now().strftime("%H:%M:%S")
        log_message = f"[{timestamp}] {message}"
        print(log_message)
        with open(self.txt_log, 'a') as f:
            f.write(log_message + "\n")

    def log_episode(self, episode, score, enemies_destroyed, survival_time, epsilon, avg_score, avg_kills):
        with open(self.csv_log, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([episode, score, enemies_destroyed, survival_time,
                           round(epsilon, 6), round(avg_score, 2), round(avg_kills, 2),
                           datetime.now().strftime("%Y-%m-%d %H:%M:%S")])

    def save_config(self, config_dict):
        with open(self.config_file, 'w') as f:
            json.dump(config_dict, f, indent=4)

    def log_summary(self, scores, enemies_destroyed, survival_times):
        duration = datetime.now() - self.start_time
        self.log("\n" + "=" * 60)
        self.log("TRAINING SUMMARY")
        self.log(f"Duration: {duration}")
        self.log(f"Episodes: {len(scores)} | Best: {max(scores)} | Avg: {np.mean(scores):.2f}")
        self.log(f"Avg Kills: {np.mean(enemies_destroyed):.2f} | Avg Survival: {np.mean(survival_times)/60:.2f}s")

print("✅ TrainingLogger defined!")

✅ TrainingLogger defined!


---
## 📌 Step 4: Game Configuration

In [ ]:
class Config:
    SCREEN_WIDTH = 600
    SCREEN_HEIGHT = 800
    PLAYER_WIDTH, PLAYER_HEIGHT = 50, 40
    PLAYER_SPEED = 8
    PLAYER_BULLET_SPEED = 12
    PLAYER_SHOOT_COOLDOWN = 15
    PLAYER_MAX_HEALTH = 3
    ENEMY_WIDTH, ENEMY_HEIGHT = 40, 35
    ENEMY_SPEED = 3
    ENEMY_BULLET_SPEED = 6
    ENEMY_SPAWN_RATE = 60
    MAX_ENEMIES = 8
    ENEMY_SHOOT_CHANCE = 0.02

    # Q-Learning hyperparameters
    LEARNING_RATE = 0.001
    DISCOUNT_FACTOR = 0.99
    EPSILON_START = 1.0
    EPSILON_MIN = 0.01
    EPSILON_DECAY = 0.999
    BATCH_SIZE = 64
    MEMORY_SIZE = 100000
    TARGET_UPDATE = 10
    STATE_SIZE = 22
    ACTION_SIZE = 6

    @classmethod
    def to_dict(cls):
        return {
            'LEARNING_RATE': cls.LEARNING_RATE,
            'DISCOUNT_FACTOR': cls.DISCOUNT_FACTOR,
            'EPSILON_START': cls.EPSILON_START,
            'EPSILON_MIN': cls.EPSILON_MIN,
            'EPSILON_DECAY': cls.EPSILON_DECAY,
            'BATCH_SIZE': cls.BATCH_SIZE,
            'MEMORY_SIZE': cls.MEMORY_SIZE,
            'TARGET_UPDATE': cls.TARGET_UPDATE,
            'STATE_SIZE': cls.STATE_SIZE,
            'ACTION_SIZE': cls.ACTION_SIZE
        }
print(f"✅ Config: State={Config.STATE_SIZE}, Actions={Config.ACTION_SIZE}, LR={Config.LEARNING_RATE}, γ={Config.DISCOUNT_FACTOR}")

✅ Config: State=22, Actions=6, LR=0.001, γ=0.99


---
## 📌 Step 5: Game Objects & Neural Network

In [ ]:
class Player:
    def __init__(self, x, y):
        self.x, self.y = x, y
        self.width, self.height = Config.PLAYER_WIDTH, Config.PLAYER_HEIGHT
        self.speed = Config.PLAYER_SPEED
        self.health = Config.PLAYER_MAX_HEALTH
        self.shoot_cooldown = 0
        self.score = 0
        self.alive = True

    def move(self, direction):
        self.x = max(0, min(self.x + direction * self.speed, Config.SCREEN_WIDTH - self.width))

    def update(self):
        if self.shoot_cooldown > 0: self.shoot_cooldown -= 1

    def can_shoot(self): return self.shoot_cooldown == 0

    def shoot(self):
        self.shoot_cooldown = Config.PLAYER_SHOOT_COOLDOWN
        return Bullet(self.x + self.width // 2 - 3, self.y - 15, -Config.PLAYER_BULLET_SPEED, True)

    def take_damage(self):
        self.health -= 1
        if self.health <= 0: self.alive = False

class Enemy:
    def __init__(self, x, y):
        self.x, self.y = x, y
        self.width, self.height = Config.ENEMY_WIDTH, Config.ENEMY_HEIGHT
        self.speed = Config.ENEMY_SPEED + random.uniform(-1, 1)
        self.health, self.alive = 1, True
        self.direction = random.choice([-1, 0, 1])
        self.change_dir_timer = random.randint(30, 90)

    def update(self):
        self.y += self.speed
        self.x = max(0, min(self.x + self.direction * 2, Config.SCREEN_WIDTH - self.width))
        self.change_dir_timer -= 1
        if self.change_dir_timer <= 0:
            self.direction = random.choice([-1, 0, 1])
            self.change_dir_timer = random.randint(30, 90)
        if self.y > Config.SCREEN_HEIGHT: self.alive = False

    def should_shoot(self): return random.random() < Config.ENEMY_SHOOT_CHANCE

    def shoot(self):
        return Bullet(self.x + self.width // 2 - 3, self.y + self.height, Config.ENEMY_BULLET_SPEED, False)

class Bullet:
    def __init__(self, x, y, speed, is_player_bullet):
        self.x, self.y, self.speed = x, y, speed
        self.width, self.height = 6, 15
        self.is_player_bullet = is_player_bullet
        self.alive = True

    def update(self):
        self.y += self.speed
        if self.y < -self.height or self.y > Config.SCREEN_HEIGHT: self.alive = False

class DQN(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, output_size)
        )
    def forward(self, x): return self.network(x)

class ReplayMemory:
    def __init__(self, capacity): self.memory = deque(maxlen=capacity)
    def push(self, *args): self.memory.append(args)
    def sample(self, batch_size): return random.sample(self.memory, batch_size)
    def __len__(self): return len(self.memory)

print("✅ Game objects and DQN defined!")

✅ Game objects and DQN defined!


---
## 📌 Step 6: Game Environment

In [ ]:
class SpaceDefenderEnv:
    def __init__(self): self.reset()

    def reset(self):
        self.player = Player(Config.SCREEN_WIDTH // 2 - Config.PLAYER_WIDTH // 2, Config.SCREEN_HEIGHT - 100)
        self.enemies, self.player_bullets, self.enemy_bullets = [], [], []
        self.frame_count = self.spawn_timer = self.enemies_destroyed = self.survival_time = 0
        self.game_over = False
        return self._get_state()

    def _get_state(self):
        state = [self.player.x / Config.SCREEN_WIDTH,
                 1.0 if self.player.can_shoot() else 0.0,
                 self.player.health / Config.PLAYER_MAX_HEALTH]

        enemies_sorted = sorted(self.enemies, key=lambda e: abs(e.x - self.player.x) + abs(e.y - self.player.y))[:3]
        for i in range(3):
            if i < len(enemies_sorted):
                e = enemies_sorted[i]
                state.extend([(e.x - self.player.x) / Config.SCREEN_WIDTH,
                             (e.y - self.player.y) / Config.SCREEN_HEIGHT,
                             1.0 if abs(e.x - self.player.x) < 100 else 0.0])
            else: state.extend([0.0, -1.0, 0.0])

        bullets_sorted = sorted(self.enemy_bullets, key=lambda b: abs(b.x - self.player.x) + abs(b.y - self.player.y))[:5]
        for i in range(5):
            if i < len(bullets_sorted):
                b = bullets_sorted[i]
                state.extend([(b.x - self.player.x) / Config.SCREEN_WIDTH,
                             (b.y - self.player.y) / Config.SCREEN_HEIGHT])
            else: state.extend([0.0, -1.0])

        return np.array(state, dtype=np.float32)

    def _check_collision(self, o1, o2):
        return o1.x < o2.x + o2.width and o1.x + o1.width > o2.x and o1.y < o2.y + o2.height and o1.y + o1.height > o2.y

    def step(self, action):
        self.frame_count += 1
        self.survival_time += 1
        reward = 0.1

        move_dir, should_shoot = 0, False
        if action == 0: move_dir = -1
        elif action == 1: move_dir = 1
        elif action == 3: should_shoot = True
        elif action == 4: move_dir, should_shoot = -1, True
        elif action == 5: move_dir, should_shoot = 1, True

        self.player.move(move_dir)
        self.player.update()
        if should_shoot and self.player.can_shoot():
            self.player_bullets.append(self.player.shoot())

        self.spawn_timer += 1
        if self.spawn_timer >= Config.ENEMY_SPAWN_RATE and len(self.enemies) < Config.MAX_ENEMIES:
            self.spawn_timer = 0
            self.enemies.append(Enemy(random.randint(0, Config.SCREEN_WIDTH - Config.ENEMY_WIDTH), -Config.ENEMY_HEIGHT))

        for e in self.enemies[:]:
            e.update()
            if e.should_shoot(): self.enemy_bullets.append(e.shoot())
            if not e.alive: self.enemies.remove(e); reward -= 0.5

        for b in self.player_bullets[:]: b.update(); (not b.alive) and self.player_bullets.remove(b)
        for b in self.enemy_bullets[:]: b.update(); (not b.alive) and self.enemy_bullets.remove(b)

        for b in self.player_bullets[:]:
            for e in self.enemies[:]:
                if self._check_collision(b, e):
                    b.alive = False; e.health -= 1; reward += 5
                    if e.health <= 0:
                        e.alive = False; self.enemies.remove(e)
                        self.enemies_destroyed += 1; self.player.score += 100; reward += 20
                    if b in self.player_bullets: self.player_bullets.remove(b)
                    break

        for b in self.enemy_bullets[:]:
            if self._check_collision(b, self.player):
                b.alive = False; self.enemy_bullets.remove(b)
                self.player.take_damage(); reward -= 30
                if not self.player.alive: self.game_over = True; reward -= 100

        for e in self.enemies[:]:
            if self._check_collision(e, self.player):
                e.alive = False; self.enemies.remove(e)
                self.player.take_damage(); reward -= 30
                if not self.player.alive: self.game_over = True; reward -= 100

        return self._get_state(), reward, self.game_over, {'score': self.player.score, 'enemies_destroyed': self.enemies_destroyed, 'survival_time': self.survival_time, 'health': self.player.health}

print("✅ SpaceDefenderEnv defined!")

✅ SpaceDefenderEnv defined!


---
## 📌 Step 7: Q-Learning Agent

In [ ]:
class QLearningAgent:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.policy_net = DQN(Config.STATE_SIZE, Config.ACTION_SIZE).to(self.device)
        self.target_net = DQN(Config.STATE_SIZE, Config.ACTION_SIZE).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=Config.LEARNING_RATE)
        self.criterion = nn.SmoothL1Loss()
        self.memory = ReplayMemory(Config.MEMORY_SIZE)
        self.epsilon = Config.EPSILON_START
        self.losses = []

    def get_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randint(0, Config.ACTION_SIZE - 1)
        with torch.no_grad():
            return self.policy_net(torch.FloatTensor(state).unsqueeze(0).to(self.device)).argmax().item()

    def remember(self, *args): self.memory.push(*args)

    def train_step(self):
        if len(self.memory) < Config.BATCH_SIZE: return
        batch = self.memory.sample(Config.BATCH_SIZE)
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.FloatTensor(np.array(states)).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(np.array(next_states)).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)

        current_q = self.policy_net(states).gather(1, actions.unsqueeze(1))
        with torch.no_grad():
            next_actions = self.policy_net(next_states).argmax(1)
            next_q = self.target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze()
            target_q = rewards + (1 - dones) * Config.DISCOUNT_FACTOR * next_q

        loss = self.criterion(current_q.squeeze(), target_q)
        self.losses.append(loss.item())
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10)
        self.optimizer.step()

    def update_target_network(self): self.target_net.load_state_dict(self.policy_net.state_dict())
    def decay_epsilon(self): self.epsilon = max(Config.EPSILON_MIN, self.epsilon * Config.EPSILON_DECAY)

    def save_model(self, path):
        torch.save({'policy_net': self.policy_net.state_dict(), 'target_net': self.target_net.state_dict(),
                   'optimizer': self.optimizer.state_dict(), 'epsilon': self.epsilon, 'losses': self.losses}, path)

    def load_model(self, path):
        ckpt = torch.load(path, map_location=self.device)
        self.policy_net.load_state_dict(ckpt['policy_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.epsilon = ckpt.get('epsilon', Config.EPSILON_MIN)

print(f"✅ QLearningAgent defined! Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

✅ QLearningAgent defined! Device: cuda


---
## 📌 Step 8: Plotting & Video Recording Functions

In [ ]:
def save_training_plots(scores, avg_scores, enemies_destroyed, survival_times, losses, filepath):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0,0].plot(scores, alpha=0.3); axes[0,0].plot(avg_scores, lw=2)
    axes[0,0].set_title('Score Progress'); axes[0,0].legend(['Score', 'Avg']); axes[0,0].grid(alpha=0.3)

    avg_d = [np.mean(enemies_destroyed[max(0,i-100):i+1]) for i in range(len(enemies_destroyed))]
    axes[0,1].plot(enemies_destroyed, alpha=0.3); axes[0,1].plot(avg_d, lw=2)
    axes[0,1].set_title('Combat Performance'); axes[0,1].grid(alpha=0.3)

    surv_s = [t/60 for t in survival_times]
    avg_s = [np.mean(surv_s[max(0,i-100):i+1]) for i in range(len(surv_s))]
    axes[1,0].plot(surv_s, alpha=0.3); axes[1,0].plot(avg_s, lw=2)
    axes[1,0].set_title('Survival Time (s)'); axes[1,0].grid(alpha=0.3)

    if losses:
        sm_loss = [np.mean(losses[max(0,i-100):i+1]) for i in range(len(losses))]
        axes[1,1].plot(sm_loss, color='red'); axes[1,1].set_title('Loss'); axes[1,1].grid(alpha=0.3)

    plt.tight_layout(); plt.savefig(filepath, dpi=150); plt.close()
    print(f"📊 Plot saved: {filepath}")

def record_gameplay(agent, num_games=3, fps=30, max_frames=1800):
    print("\n🎬 Recording AI gameplay...")
    from pyvirtualdisplay import Display
    display = Display(visible=0, size=(Config.SCREEN_WIDTH, Config.SCREEN_HEIGHT)); display.start()

    import pygame; pygame.init()
    screen = pygame.display.set_mode((Config.SCREEN_WIDTH, Config.SCREEN_HEIGHT))
    font = pygame.font.SysFont('arial', 24)
    DARK_BLUE, WHITE, BLUE, CYAN, RED, ORANGE, YELLOW, GREEN = (10,10,40), (255,255,255), (50,150,255), (0,255,255), (255,50,50), (255,165,0), (255,255,0), (50,255,50)

    all_frames = []
    for g in range(num_games):
        print(f"  Game {g+1}/{num_games}...")
        env = SpaceDefenderEnv(); state = env.reset(); done = False; frames = []; fc = 0

        while not done and fc < max_frames:
            action = agent.get_action(state, training=False)
            state, _, done, info = env.step(action)
            screen.fill(DARK_BLUE)

            for _ in range(50): pygame.draw.circle(screen, WHITE, (random.randint(0,Config.SCREEN_WIDTH), random.randint(0,Config.SCREEN_HEIGHT)), 1)

            p = env.player
            pygame.draw.polygon(screen, BLUE, [(p.x+p.width//2,p.y), (p.x+p.width,p.y+p.height), (p.x+p.width//2,p.y+p.height-10), (p.x,p.y+p.height)])
            if fc%4<2: pygame.draw.polygon(screen, ORANGE, [(p.x+10,p.y+p.height), (p.x+p.width//2,p.y+p.height+15), (p.x+p.width-10,p.y+p.height)])

            for e in env.enemies:
                pygame.draw.polygon(screen, RED, [(e.x,e.y), (e.x+e.width,e.y), (e.x+e.width//2,e.y+e.height)])

            for b in env.player_bullets: pygame.draw.rect(screen, CYAN, (b.x,b.y,b.width,b.height))
            for b in env.enemy_bullets: pygame.draw.rect(screen, RED, (b.x,b.y,b.width,b.height))

            screen.blit(font.render(f"Score: {info['score']}", True, WHITE), (10,10))
            screen.blit(font.render(f"Kills: {info['enemies_destroyed']}", True, WHITE), (10,40))
            for i in range(env.player.health): pygame.draw.rect(screen, GREEN, (10+i*25,70,20,20))
            screen.blit(font.render("AI PLAYING", True, GREEN), (Config.SCREEN_WIDTH-150,10))

            pygame.display.flip()
            frame = np.transpose(pygame.surfarray.array3d(screen), (1,0,2)); frames.append(frame); fc += 1

        print(f"    Score: {info['score']} | Kills: {info['enemies_destroyed']}")
        all_frames.extend(frames)
        for _ in range(fps): all_frames.append(frames[-1] if frames else np.zeros((Config.SCREEN_HEIGHT,Config.SCREEN_WIDTH,3), dtype=np.uint8))

    pygame.quit(); display.stop()

    video_path = f"{DIRS['videos']}/ai_gameplay.mp4"
    imageio.mimsave(video_path, all_frames, fps=fps)
    imageio.mimsave(f"{DIRS['base']}/ai_gameplay.mp4", all_frames, fps=fps)
    print(f"✅ Video saved: {video_path}")
    return video_path

print("✅ Plotting and video functions defined!")

✅ Plotting and video functions defined!


---
## 📌 Step 9: Training Function

In [ ]:
def train(episodes=1000, save_interval=100, print_interval=25):
    logger = TrainingLogger(DIRS['logs'])
    logger.save_config(Config.to_dict())

    agent = QLearningAgent()
    env = SpaceDefenderEnv()
    scores, avg_scores, enemies_destroyed_list, survival_times = [], [], [], []
    best_score = 0

    logger.log(f"SPACE DEFENDER TRAINING | Episodes: {episodes} | Device: {agent.device}")

    for ep in range(episodes):
        state = env.reset(); done = False; steps = 0
        while not done and steps < 3000:
            action = agent.get_action(state, True)
            next_state, reward, done, info = env.step(action)
            agent.remember(state, action, reward, next_state, done)
            agent.train_step()
            state = next_state; steps += 1

        if ep % Config.TARGET_UPDATE == 0: agent.update_target_network()
        agent.decay_epsilon()

        scores.append(info['score']); enemies_destroyed_list.append(info['enemies_destroyed']); survival_times.append(info['survival_time'])
        avg_score = np.mean(scores[-100:]); avg_scores.append(avg_score); avg_kills = np.mean(enemies_destroyed_list[-100:])

        logger.log_episode(ep+1, info['score'], info['enemies_destroyed'], info['survival_time'], agent.epsilon, avg_score, avg_kills)

        if info['score'] > best_score:
            best_score = info['score']
            agent.save_model(f"{DIRS['best_model']}/best_model.pth")
            logger.log(f"🏆 New best: {best_score}!")

        if (ep+1) % print_interval == 0:
            logger.log(f"Ep {ep+1:4d}/{episodes} | Score: {info['score']:5d} | Avg: {avg_score:7.1f} | Best: {best_score:5d} | ε: {agent.epsilon:.4f}")

        if (ep+1) % save_interval == 0:
            agent.save_model(f"{DIRS['models']}/checkpoint_ep{ep+1}.pth")
            save_training_plots(scores, avg_scores, enemies_destroyed_list, survival_times, agent.losses, f"{DIRS['plots']}/progress_ep{ep+1}.png")

    logger.log_summary(scores, enemies_destroyed_list, survival_times)
    agent.save_model(f"{DIRS['models']}/final_model.pth")
    save_training_plots(scores, avg_scores, enemies_destroyed_list, survival_times, agent.losses, f"{DIRS['base']}/training_progress.png")

    return agent, scores, avg_scores, enemies_destroyed_list, survival_times

print("✅ Training function ready!")

✅ Training function ready!


---
# 🚀 RUN TRAINING

In [ ]:
# ============================================================
# 🎮 START TRAINING - Adjust episodes as needed
# ============================================================

NUM_EPISODES = 1000  # Recommend 500-1000 for good results

print(f"🚀 Training {NUM_EPISODES} episodes...")
print(f"📁 Saving to: {DIRS['run']}")

trained_agent, scores, avg_scores, enemies_destroyed, survival_times = train(
    episodes=NUM_EPISODES,
    save_interval=100,
    print_interval=25
)

print("\n✅ TRAINING COMPLETE!")

🚀 Training 1000 episodes...
📁 Saving to: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833
[04:00:23] Training started at 2026-01-13 04:00:23.674875
[04:00:29] SPACE DEFENDER TRAINING | Episodes: 1000 | Device: cuda
[04:00:34] 🏆 New best: 800!
[04:00:42] 🏆 New best: 900!
[04:00:45] 🏆 New best: 1000!
[04:01:02] 🏆 New best: 1200!
[04:01:42] 🏆 New best: 1400!
[04:01:55] Ep   25/1000 | Score:   300 | Avg:   620.0 | Best:  1400 | ε: 0.9876
[04:02:57] Ep   50/1000 | Score:   500 | Avg:   496.0 | Best:  1400 | ε: 0.9753
[04:04:12] Ep   75/1000 | Score:   600 | Avg:   498.7 | Best:  1400 | ε: 0.9632
[04:05:42] Ep  100/1000 | Score:   600 | Avg:   530.0 | Best:  1400 | ε: 0.9512
📊 Plot saved: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833/plots/progress_ep100.png
[04:06:48] 🏆 New best: 1500!
[04:07:07] Ep  125/1000 | Score:   500 | Avg:   515.0 | Best:  1500 | ε: 0.9394
[04:07:57] 🏆 New best: 1600!
[04:08:41] Ep  150/1000 | Sc

---
# 🎬 RECORD VIDEO

In [ ]:
# Record 3 games of AI gameplay
video_path = record_gameplay(trained_agent, num_games=3, fps=30, max_frames=1800)
print(f"\n🎬 Video ready at: {video_path}")


🎬 Recording AI gameplay...
pygame 2.6.1 (SDL 2.28.4, Python 3.12.12)
Hello from the pygame community. https://www.pygame.org/contribute.html
  Game 1/3...
    Score: 0 | Kills: 0
  Game 2/3...
    Score: 0 | Kills: 0
  Game 3/3...
    Score: 0 | Kills: 0


✅ Video saved: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833/videos/ai_gameplay.mp4

🎬 Video ready at: /content/drive/MyDrive/Assignment 1 - Norman Lee/training_runs/run_20260113_035833/videos/ai_gameplay.mp4


---
# 📊 VIEW RESULTS

In [ ]:
from IPython.display import Image, display
display(Image(filename=f"{DIRS['base']}/training_progress.png"))

print(f"\n📁 ALL FILES SAVED TO: {DIRS['base']}")
for name, path in DIRS.items():
    if os.path.exists(path):
        files = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
        if files: print(f"  {name}: {len(files)} files")